# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharl20/Flyrank_01/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

What performance archetypes exist across a content inventory, and how should a content team prioritize action on them?

This supports one decision: which pages a content strategist or editor should review first in a given sprint — protect, improve, rewrite, monitor — using only structured, safe metrics (traffic, engagement, freshness, token counts). No article text was used, so this is behavioral clustering, not semantic clustering. The work is decision-support for a human reviewer, not an automated content system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Data
Source: FlyRank ML Internship starter dataset — data/raw/content_refresh_anonymized.csv, 30,000 rows, one row per pseudonymized content page, 32 clients, trailing-90-day metrics.

Date window: trailing 90 days from extract date, with layered sub-windows (*_last_30d, *_prev_30d for trend calculation).

Excluded fields, with why:

trend_direction / trend_pct as ML features (though used as a rule input for the baseline) — these are derived the same way a label would be, so using them as model inputs would be circular.
provider_used / model_used — production/tooling metadata, not performance signal; mixing it in risks clusters reflecting the content pipeline rather than audience behavior.
content_id / client_id — identifiers, used only for grouping/splitting, never as features.
No client names, raw URLs, or query strings appear anywhere in this dataset or this paper — all identifiers are pseudonyms per the dataset's own design.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/JoeSherif97/FlyRankerML/"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


In [2]:
print(f"Rows: {len(df)} | Unique clients: {df['client_id'].nunique()} | Unique pages: {df['content_id'].nunique()}")

Rows: 30000 | Unique clients: 32 | Unique pages: 30000


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Task type: Clustering (unsupervised) — "what kinds of pages exist," not "will this page decline." No observed future outcome exists in this snapshot, so classification was not honest here.

Label: none. Archetype names (champion, hidden gem, weak/no-demand) are assigned to clusters after inspecting their contents, not defined in advance.

Baseline: a transparent, human-readable rule (stale_declining_visible, declining_visible_fresh, stale_visible_stable, low_priority), scored as declining × visible × (1 + stale) × impressions_90d, with one reason code per page.

Features: structured 90-day metrics (search economics, performance, engagement, freshness) plus has_-flags for four fields confirmed to be missing by content_type rather than randomly (search_volume, competition, cpc, word_count).

Validation design: grouped split by client_id (GroupShuffleSplit), not random — client sizes are highly skewed (largest client = 23% of rows), so a random split lets the model memorize a client's own scale rather than generalize.

Leakage checks: full checklist run (label-derived fields, product-decision flags, ID columns, group overlap) plus a deliberate-injection test — adding a known-leaky field (trend_pct) measurably raised holdout silhouette (0.278 → 0.293), confirming the audit harness was actually sensitive to leakage rather than rubber-stamping a clean-looking list.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [3]:


comparison_table = pd.DataFrame({
    "Metric": [
        "Groups found",
        "Resolution inside baseline's largest bucket (21,085 rows)",
        "Holdout silhouette — random split",
        "Holdout silhouette — grouped-by-client split (honest)",
        "Holdout silhouette — with deliberately leaked feature",
    ],
    "Baseline (4 hand-written buckets)": ["4", "1 undifferentiated bucket", "—", "—", "—"],
    "K-Means (k=4)": ["4", "4 distinct sub-clusters", "0.294", "0.278", "0.293 (confirms leakage detection works)"],
})
print(comparison_table.to_string(index=False))

                                                   Metric Baseline (4 hand-written buckets)                            K-Means (k=4)
                                             Groups found                                 4                                        4
Resolution inside baseline's largest bucket (21,085 rows)         1 undifferentiated bucket                  4 distinct sub-clusters
                        Holdout silhouette — random split                                 —                                    0.294
    Holdout silhouette — grouped-by-client split (honest)                                 —                                    0.278
    Holdout silhouette — with deliberately leaked feature                                 — 0.293 (confirms leakage detection works)



K-Means found real but partial structure beyond the baseline's four buckets — most clearly a 382-page high-performing cluster (median 503 sessions, 2.63% engagement, page-1 position) and a 2,469-page near-invisible cluster (median 6 impressions). However, a shallow decision tree reading the clusters back showed the split leaned heavily on has_word_count (importance 0.271) — a near-proxy for content_type — meaning part of the clustering rediscovers known content categories rather than finding a wholly new behavioral pattern. This is stated as a limitation, not smoothed over.

## 5. Limitations

*What this work cannot claim.*

No article text — clustering is behavioral/structural, not semantic. Archetype names describe traffic and engagement shape, not topic or meaning.
Content-type confound — the strongest driver of cluster split (has_word_count) is a near-proxy for content_type, so part of the "archetype" signal is closer to content category than genuine behavioral discovery.
No validated future outcome — the baseline rule uses trend_direction, itself a defined rule (30d vs. prev 30d), not an independently observed future result. No precision@K against a real future window exists in this snapshot.
Client concentration — one client holds 23% of rows; the priority queue's top candidates are disproportionately drawn from a small number of large clients, confirmed directly rather than assumed.
Small holdout for grouped validation — the grouped split's holdout covers only 7 of 32 clients; a single unusual client could swing the reported silhouette more than a larger holdout would.
Cross-sectional, one snapshot — no causal claims are supported. This is observed, directional, decision-support structure only.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [4]:

import os
print("cwd:", os.getcwd())
print("work/ exists:", os.path.isdir("work"))
print("work/outputs/ exists:", os.path.isdir("work/outputs"))
if os.path.isdir("work/outputs"):
    print("contents:", os.listdir("work/outputs"))

cwd: /content/flyrank-ml-internship-starter
work/ exists: True
work/outputs/ exists: True
contents: ['monitoring_baseline.json', 'human_review_sample.csv', 'playbook_summary_by_action.csv', 'content_action_playbook.csv']


In [5]:
# pulled directly from the ML-10 playbook export
playbook = pd.read_csv("work/outputs/content_action_playbook.csv")
print(playbook["action"].value_counts())
print(playbook.groupby("action", observed=True)[["impressions_90d", "priority_score"]].median())


action
improve                14907
monitor                12065
protect_and_refresh     3015
rewrite                   13
Name: count, dtype: int64
                     impressions_90d  priority_score
action                                              
improve                        780.0           656.0
monitor                        447.0             0.0
protect_and_refresh          15342.0           816.0
rewrite                       4556.0          9112.0


Protect and refresh champions first (3,015 pages) — highest-value pages, often also going stale; losing these costs the most.
Rewrite the small, high-confidence stale-and-declining group (13 pages) — still visible, losing ground, untouched 180+ days.
Improve the larger declining-but-fresh group (14,907 pages) — investigate why a recently updated page is still declining before assuming a rewrite fixes it.
Monitor the rest (12,065 pages) — includes weak/no-demand pages where avg_position is statistically unreliable due to tiny impression bases; do not prune on position alone.

No action here should be automated end-to-end — see the human-review and no-go list in the ML-10 notebook for what a person must check before acting on any recommendation.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [6]:


import matplotlib.pyplot as plt

# Chart 1: action queue distribution
action_counts = playbook["action"].value_counts()
fig, ax = plt.subplots(figsize=(6,4))
action_counts.plot(kind="bar", ax=ax, color="#2F6F6B")
ax.set_title("Pages per recommended action")
ax.set_ylabel("Page count")
plt.tight_layout()
plt.savefig("work/outputs/fig_action_distribution.png", dpi=150)
plt.close()

# Chart 2: baseline vs clustering resolution inside low_priority
comparison_table.to_csv("work/outputs/table_model_vs_baseline.csv", index=False)

print("Saved: fig_action_distribution.png, table_model_vs_baseline.csv")

Saved: fig_action_distribution.png, table_model_vs_baseline.csv


In [7]:

import os
os.makedirs("docs", exist_ok=True)
print("docs/ exists:", os.path.isdir("docs"))

docs/ exists: True


In [8]:

import shutil
shutil.copy("work/outputs/fig_action_distribution.png", "docs/fig_action_distribution.png")

'docs/fig_action_distribution.png'

In [9]:

%%writefile docs/index.html





Structured Content Archetype Clustering — A Portfolio Lifecycle Study








    Data Report · FlyRank ML Internship Capstone
    Structured Content Archetype Clustering

      What performance archetypes exist across a content inventory — and which pages should an
      editor act on first? A behavioral clustering study on 30,000 pages, using only structured
      metrics: no article text, no semantic claims, decision-support only.




        65.7%
        of sessions from the top 10% of pages


        54.2%
        of pages trending down over 30 days


        26.9%
        low-traffic pages with above-median engagement


        0.278
        holdout silhouette, honest client-grouped split







    Abstract
    Introduction
    Data
    Methodology
    Results
    Limitations
    Recommendations
    Reproducibility
    Data credit






    01
    Abstract

      Content teams managing large page inventories have no systematic way to see which pages
      behave alike, relying instead on gut feel and single-metric sorts. This study used K-Means
      clustering on 30,000 structured content records — 90-day traffic, engagement, and freshness
      metrics, with no article text — to test whether recurring behavioral archetypes exist beyond
      a hand-written priority rule. Clustering found real but partial structure: a small,
      high-value "champion" cluster and a near-invisible "weak-demand" cluster stood out clearly,
      while the remaining two clusters were driven substantially by content type rather than
      independent behavior. A grouped-by-client validation split showed the result was not simply
      client memorization (holdout silhouette 0.278 vs. 0.294 on a random split), and a deliberate
      leakage test confirmed the audit process was genuinely sensitive to contamination. The
      output is a ranked, reason-coded action queue intended for human review — decision-support,
      not an automated recommendation engine, and not a claim about causation or future performance.




    02
    Introduction

      A content strategist facing a backlog of thousands of pages needs to answer one question
      every sprint: where should limited editor time go first? Single-metric sorts —
      "top 10 by traffic" — miss interaction effects: a low-traffic page can be a hidden gem if
      engagement is high, or genuinely dead weight if it isn't. A wrong call is not free. Pruning a
      quietly-working page wastes rebuild cost later; leaving a declining page alone because it
      still looks strong on one metric compounds lost traffic; rewriting a page that's actually
      competing with a sibling page wastes hours on the wrong fix.


      This study asks whether unsupervised clustering — grouping pages by structural similarity,
      not by a predicted outcome — can add resolution beyond what a transparent, hand-written rule
      already provides, and states plainly where that added resolution is genuine versus where it
      is closer to rediscovering known content categories.




    03
    Data

      Source: the FlyRank ML Internship starter release — 30,000 pseudonymized content pages
      across 32 clients, trailing-90-day Google Search Console and GA4-derived metrics. No client
      names, raw URLs, or private queries appear anywhere in this dataset; all identifiers are
      pseudonyms by design.


      FieldRoleWhy
      content_id, client_idContextGrouping and splitting only, never a feature
      trend_direction, trend_pctExcluded as featureDerived the same way a label would be — using it risks circularity
      provider_used, model_usedExcludedProduction metadata, not audience-behavior signal
      15 structured metrics + 4 missingness flagsFeatureSafe, structured, knowable at analysis time


      Public-safe note
      Missingness in search_volume, word_count
      and related fields was confirmed to follow content_type almost
      deterministically (e.g. 100% of one content type structurally lacks keyword-volume data) —
      handled with explicit has_-flags rather than a blind fill, to avoid
      injecting a fabricated signal.




    04
    Methodology
    Task framing

      Clustering, not classification: no observed future outcome exists in this snapshot to
      predict, so the honest task is "what kinds of pages exist," evaluated by silhouette score
      plus a human sense-check of whether cluster profiles match recognizable archetypes.

    Baseline

      A transparent, readable rule assigns every page one reason code
      (stale_declining_visible, declining_visible_fresh,
      stale_visible_stable, low_priority), scored as
      declining × visible × (1 + stale) × impressions_90d — no fitted weights,
      fully human-auditable.

    Validation design

      Split grouped by client_id, not random. Client size is highly
      skewed (largest client holds 23% of all rows), so a random split lets the model implicitly
      memorize a client's own scale. A seed-sensitivity check across five seeds confirmed this
      matters: the resulting split ratio swung from 66/34 to 93/7 despite requesting 80/20 every
      time, purely from which large client landed in holdout.

    Leakage audit

      Full checklist run against the final 19-field feature set: no label-derived fields, no
      product-decision flags, no identifier columns, zero client overlap between fit and holdout.
      A deliberate-injection test — adding trend_pct, a field known to be
      derived the same way as the excluded label — raised holdout silhouette from 0.278 to 0.293,
      confirming the audit harness genuinely detects leakage rather than passing by default.




    05
    Results



        Pages per recommended action, from the final ranked queue. monitor
        and improve hold the bulk of the portfolio; the small
        rewrite bucket (13 pages) is the highest-confidence,
        highest-urgency group — visible, declining, and stale for 180+ days.




      MetricBaseline (4 hand-written buckets)K-Means (k=4)
      Groups found44
      Resolution inside baseline's largest bucket (21,085 rows)1 undifferentiated bucketSplit across all 4 clusters
      Validated metricNone (rule-based)Silhouette = 0.278 (holdout)



      Read this carefully
      A decision tree explaining the clusters found that has_word_count
      — a near-perfect proxy for content_type — was the dominant driver
      (importance 0.271) of cluster assignment, ahead of any performance signal. Two of the four
      clusters are genuinely distinct behavioral groups (a 382-page champion cluster and a
      2,469-page weak-demand cluster); the other two are better described as
      "content-type-driven, with performance sub-splits inside," not four independently discovered
      archetypes.




    06
    Limitations

      No article text. Clustering is behavioral, not semantic — archetype names describe traffic and engagement shape, not topic or meaning.
      Content-type confound. The strongest cluster-driving feature is a near-proxy for content type, not a purely new behavioral signal.
      No validated future outcome. The baseline's "declining" label is a defined rule, not an independently observed result — no precision@K against a real future window exists in this snapshot.
      Client concentration. One client holds 23% of rows; the priority queue's top candidates are disproportionately drawn from a small number of large clients.
      Small grouped holdout. Only 7 of 32 clients fall in the validation holdout; one unusual client could swing the reported silhouette.
      Cross-sectional data, one snapshot. No causal claims are supported anywhere in this work — only observed, directional, decision-support structure.




    07
    Ranked recommendations
    The output of this work is a reason-coded action queue for human review, not an automated system:

      01Protect and refresh champions first. The highest-value pages are often also the most neglected relative to their value — losing one costs the most.
      02Rewrite the small, high-confidence stale-and-declining group. Still visible, losing ground, untouched 180+ days — the clearest actionable signal in the queue.
      03Improve the larger declining-but-fresh group. A page that's still declining despite a recent update needs investigation, not an assumed rewrite.
      04Monitor the rest. Includes weak-demand pages where average position is statistically unreliable due to a tiny impression base — do not prune on position alone.


      What should never be automated
      Auto-publishing or auto-editing content from this queue; auto-pruning or auto-merging pages
      based on a "monitor" label; committing budget from the priority score as if it were a
      validated economic model; or treating a page's cluster/archetype as a permanent identity fed
      into other systems. Every recommendation here requires human review before action.




    08
    Reproducibility

      All notebooks live in this repository under work/notebooks/. The
      exported action queue, review sample, and monitoring baseline are in
      work/outputs/. Random seeds are fixed and stated in each notebook;
      rerunning top-to-bottom reproduces every table and figure on this page.




    09
    Acknowledgments & data credit

      Built on the FlyRank ML Internship dataset. Data credit and program information:
      flyrank.ai.







    Structured Content Archetype Clustering — a decision-support study, not a causal claim. Observed, directional, decision-support only.







Overwriting docs/index.html


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.